## Setup and Imports

In [1]:
# Standard library imports
import sys
from pathlib import Path
from datetime import datetime

# Third-party imports
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Project imports
from ayne.utils.query_utils import (
    get_database_summary_stats,
    get_enrichment_status_by_year,
    get_movies_due_for_update_by_year,
    get_data_quality_metrics,
    get_recent_update_activity,
    get_refresh_compliance_by_age_group,
    get_consecutive_unchanged_stats,
    get_data_freshness_by_year,
    get_api_usage_estimates,
    execute_custom_query
)

# Notebook dark mode utilities
from ayne.utils.plotting import configure_dark_mode

# Configure dark mode for the notebook
configure_dark_mode()

# Display configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print(f"✓ Setup complete - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Setup complete - 2025-12-04 19:26:40


## 1. Database Overview

High-level statistics about the entire database.

In [2]:
# Get summary statistics
stats = get_database_summary_stats()

# Display as formatted output
print("="*60)
print(" " * 15 + "DATABASE SUMMARY STATISTICS")
print("="*60)
print(f"\n📊 Total Movies: {stats['total_movies']:,}")
print(f"   └─ Date Range: {stats['earliest_movie']} to {stats['latest_movie']}")
print(f"   └─ Unique Years: {stats['unique_years']}")
print(f"\n🔑 ID Coverage:")
print(f"   └─ With TMDB ID: {stats['with_tmdb_id']:,} ({100.0 * stats['with_tmdb_id'] / stats['total_movies']:.1f}%)")
print(f"   └─ With IMDB ID: {stats['with_imdb_id']:,} ({100.0 * stats['with_imdb_id'] / stats['total_movies']:.1f}%)")
print(f"\n✨ Enrichment Status:")
print(f"   └─ TMDB Enriched: {stats['enriched_tmdb']:,} ({stats['tmdb_enrichment_pct']:.1f}%)")
print(f"   └─ OMDB Enriched: {stats['enriched_omdb']:,} ({stats['omdb_enrichment_pct']:.1f}%)")
print(f"   └─ Fully Refreshed: {stats['fully_refreshed']:,} ({100.0 * stats['fully_refreshed'] / stats['total_movies']:.1f}%)")
print(f"\n❄️  Frozen Movies: {stats['frozen_movies']:,}")
print("="*60)

               DATABASE SUMMARY STATISTICS

📊 Total Movies: 32,534
   └─ Date Range: 1950-01-01 00:00:00 to 2025-11-16 00:00:00
   └─ Unique Years: 76

🔑 ID Coverage:
   └─ With TMDB ID: 32,534 (100.0%)
   └─ With IMDB ID: 32,245 (99.1%)

✨ Enrichment Status:
   └─ TMDB Enriched: 32,534 (100.0%)
   └─ OMDB Enriched: 6,663 (20.5%)
   └─ Fully Refreshed: 6,663 (20.5%)

❄️  Frozen Movies: 0


## 2. Enrichment Status by Year

Shows how many movies per year have:
- Base IDs (TMDB/IMDB)
- Full TMDB enrichment
- OMDB enrichment

In [3]:
# Get enrichment breakdown by year
enrichment_df = get_enrichment_status_by_year()

# Display summary
print(f"\n📅 Enrichment Status for {len(enrichment_df)} Years\n")
print(enrichment_df.to_string(index=False))

# Show recent years in detail
print(f"\n\n🔍 Recent Years (2020+):\n")
recent = enrichment_df[enrichment_df['release_year'] >= 2020].copy()
if not recent.empty:
    print(recent.to_string(index=False))
else:
    print("No data for recent years yet.")


📅 Enrichment Status for 76 Years

 release_year  total_movies  with_base_ids  enriched_tmdb  enriched_omdb  pct_tmdb_enriched  pct_omdb_enriched
         2025           397            397            397            258              100.0              64.99
         2024           791            791            791            714              100.0              90.27
         2023           984            984            984            978              100.0              99.39
         2022          1083           1083           1083           1073              100.0              99.08
         2021          1092           1092           1092           1087              100.0              99.54
         2020          1038           1038           1038           1010              100.0              97.30
         2019          1353           1353           1353           1349              100.0              99.70
         2018          1390           1390           1390            194     

In [4]:
# Visualize enrichment status
if len(enrichment_df) > 0:
    # Filter to years with data (optional: adjust range as needed)
    plot_df = enrichment_df[enrichment_df['release_year'] >= 1980].copy()

    fig = go.Figure()

    # Add traces for each enrichment level
    fig.add_trace(go.Scatter(
        x=plot_df['release_year'],
        y=plot_df['total_movies'],
        name='Total Movies',
        mode='lines+markers',
        line=dict(width=2, color='lightgray'),
        marker=dict(size=6)
    ))

    fig.add_trace(go.Scatter(
        x=plot_df['release_year'],
        y=plot_df['enriched_tmdb'],
        name='TMDB Enriched',
        mode='lines+markers',
        line=dict(width=2, color='#01b4e4'),  # TMDB blue
        marker=dict(size=6)
    ))

    fig.add_trace(go.Scatter(
        x=plot_df['release_year'],
        y=plot_df['enriched_omdb'],
        name='OMDB Enriched',
        mode='lines+markers',
        line=dict(width=2, color='#f5c518'),  # IMDB yellow
        marker=dict(size=6)
    ))

    fig.update_layout(
        title='Data Enrichment Status by Year',
        xaxis_title='Release Year',
        yaxis_title='Number of Movies',
        hovermode='x unified',
        height=500
    )

    fig.show()
else:
    print("No data available for visualization.")

In [5]:
# Enrichment percentage heatmap (recent years)
if len(enrichment_df) > 0:
    recent_years = enrichment_df[enrichment_df['release_year'] >= 2000].copy()

    if not recent_years.empty:
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=('TMDB Enrichment %', 'OMDB Enrichment %')
        )

        fig.add_trace(
            go.Bar(
                x=recent_years['release_year'],
                y=recent_years['pct_tmdb_enriched'],
                name='TMDB %',
                marker_color='#01b4e4'
            ),
            row=1, col=1
        )

        fig.add_trace(
            go.Bar(
                x=recent_years['release_year'],
                y=recent_years['pct_omdb_enriched'],
                name='OMDB %',
                marker_color='#f5c518'
            ),
            row=1, col=2
        )

        fig.update_yaxes(title_text='Percentage', range=[0, 105], row=1, col=1)
        fig.update_yaxes(title_text='Percentage', range=[0, 105], row=1, col=2)
        fig.update_xaxes(title_text='Year', row=1, col=1)
        fig.update_xaxes(title_text='Year', row=1, col=2)

        fig.update_layout(
            title_text='Enrichment Completion Rates (2000+)',
            showlegend=False,
            height=400
        )

        fig.show()

## 3. Movies Due for Update

Shows movies that need refreshing based on `movie_refresh_state` table.

In [6]:
# Get movies due for update by year
update_df = get_movies_due_for_update_by_year()

if len(update_df) > 0:
    total_due = update_df['movies_due_for_update'].sum()
    total_never = update_df['never_refreshed'].sum()
    total_overdue = update_df['overdue'].sum()

    print("="*60)
    print(" " * 15 + "UPDATE REQUIREMENTS")
    print("="*60)
    print(f"\n🔄 Total Movies Due for Update: {total_due:,}")
    print(f"   └─ Never Refreshed: {total_never:,}")
    print(f"   └─ Overdue: {total_overdue:,}")
    print("\n" + "="*60)

    print(f"\n📋 Breakdown by Year:\n")
    print(update_df.to_string(index=False))

    # Show top years needing updates
    top_10 = update_df.nlargest(10, 'movies_due_for_update')
    print(f"\n\n🔝 Top 10 Years Needing Updates:\n")
    print(top_10.to_string(index=False))
else:
    print("✅ All movies are up to date!")

               UPDATE REQUIREMENTS

🔄 Total Movies Due for Update: 32,534
   └─ Never Refreshed: 32,534
   └─ Overdue: 0


📋 Breakdown by Year:

 release_year  movies_due_for_update  never_refreshed  overdue
         2025                    397              397        0
         2024                    791              791        0
         2023                    984              984        0
         2022                   1083             1083        0
         2021                   1092             1092        0
         2020                   1038             1038        0
         2019                   1353             1353        0
         2018                   1390             1390        0
         2017                   1371             1371        0
         2016                   1285             1285        0
         2015                   1222             1222        0
         2014                   1179             1179        0
         2013                   1086

In [7]:
# Visualize update requirements
if len(update_df) > 0 and update_df['movies_due_for_update'].sum() > 0:
    # Filter to years with updates needed
    plot_df = update_df[update_df['movies_due_for_update'] > 0].copy()

    if len(plot_df) > 0:
        fig = go.Figure()

        fig.add_trace(go.Bar(
            x=plot_df['release_year'],
            y=plot_df['never_refreshed'],
            name='Never Refreshed',
            marker_color='#ff6b6b'
        ))

        fig.add_trace(go.Bar(
            x=plot_df['release_year'],
            y=plot_df['overdue'],
            name='Overdue',
            marker_color='#ffa726'
        ))

        fig.update_layout(
            title='Movies Requiring Updates by Year',
            xaxis_title='Release Year',
            yaxis_title='Number of Movies',
            barmode='stack',
            height=500,
            hovermode='x unified'
        )

        fig.show()
    else:
        print("No movies currently need updates.")
else:
    print("✅ No updates needed - database is current!")

## 4. Data Quality Metrics

Field completeness across different tables.

In [8]:
# Get data quality metrics
quality_df = get_data_quality_metrics()

print("\n📈 Data Quality Metrics (Field Completeness %)\n")
print(quality_df.to_string(index=False))

# Visualize quality metrics
if len(quality_df) > 0:
    # Melt dataframe for easier plotting
    quality_melted = quality_df.melt(
        id_vars=['source_table', 'total_records'],
        var_name='metric',
        value_name='completeness_pct'
    )

    fig = px.bar(
        quality_melted,
        x='metric',
        y='completeness_pct',
        color='source_table',
        barmode='group',
        title='Data Quality: Field Completeness by Table',
        labels={'completeness_pct': 'Completeness (%)', 'metric': 'Field'},
        height=500
    )

    fig.update_layout(yaxis_range=[0, 105])
    fig.add_hline(y=90, line_dash="dash", line_color="green", annotation_text="Target: 90%")
    fig.show()


📈 Data Quality Metrics (Field Completeness %)

source_table  total_records  title_completeness  release_date_completeness  tmdb_id_completeness  imdb_id_completeness
Movies Table          32534               100.0                     100.00                100.00                 99.11
 TMDB Movies          32534               100.0                      99.80                 37.33                 41.50
 OMDB Movies           6755               100.0                      95.87                 99.76                 99.45


## 5. Recent Update Activity

Track database update activity over the last 14 days.

In [9]:
# Get recent update activity
activity_df = get_recent_update_activity(days=14)

print("\n📊 Update Activity (Last 14 Days)\n")
print(activity_df.to_string(index=False))

# Summary stats
total_tmdb = activity_df['tmdb_updates'].sum()
total_omdb = activity_df['omdb_updates'].sum()
total_full = activity_df['full_refreshes'].sum()

print(f"\n📈 14-Day Summary:")
print(f"   └─ TMDB Updates: {total_tmdb:,}")
print(f"   └─ OMDB Updates: {total_omdb:,}")
print(f"   └─ Full Refreshes: {total_full:,}")


📊 Update Activity (Last 14 Days)

      date  tmdb_updates  omdb_updates  full_refreshes
2025-12-04             0           988             988
2025-12-03             0             0               0
2025-12-02             0           976             976
2025-12-01             0           916             916
2025-11-30         28767             0               0
2025-11-29            18           896             896
2025-11-28             0           989             989
2025-11-27          3749           898             898
2025-11-26             0             0               0
2025-11-25             0           997             997
2025-11-24             0             0               0
2025-11-23             0             3               3
2025-11-22             0             0               0
2025-11-21             0             0               0

📈 14-Day Summary:
   └─ TMDB Updates: 32,534
   └─ OMDB Updates: 6,663
   └─ Full Refreshes: 6,663


In [10]:
# Visualize update activity
if len(activity_df) > 0:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=activity_df['date'],
        y=activity_df['tmdb_updates'],
        name='TMDB Updates',
        mode='lines+markers',
        line=dict(width=2, color='#01b4e4')
    ))

    fig.add_trace(go.Scatter(
        x=activity_df['date'],
        y=activity_df['omdb_updates'],
        name='OMDB Updates',
        mode='lines+markers',
        line=dict(width=2, color='#f5c518')
    ))

    fig.add_trace(go.Scatter(
        x=activity_df['date'],
        y=activity_df['full_refreshes'],
        name='Full Refreshes',
        mode='lines+markers',
        line=dict(width=2, color='#10b981')
    ))

    fig.update_layout(
        title='Database Update Activity (Last 14 Days)',
        xaxis_title='Date',
        yaxis_title='Number of Updates',
        hovermode='x unified',
        height=450
    )

    fig.show()


## 6. Refresh Strategy Compliance

Checks if movies are being refreshed according to their age-based intervals:
- **Recent (0-60d)**: Every 5 days
- **New (61-180d)**: Every 14 days
- **Established (181-730d)**: Every 30 days
- **Mature (731-1825d)**: Every 90 days
- **Archived (1826+d)**: Every 180 days

In [11]:
# Get refresh compliance data
compliance_df = get_refresh_compliance_by_age_group()

print("\n📋 Refresh Strategy Compliance Analysis\n")
print(compliance_df.to_string(index=False))

# Calculate overall compliance
if not compliance_df.empty:
    total_compliant = compliance_df['compliant_tmdb'].sum()
    total_checked = compliance_df['total_movies'].sum() - compliance_df['never_updated_tmdb'].sum()
    overall_pct = 100.0 * total_compliant / total_checked if total_checked > 0 else 0

    print(f"\n📊 Overall Compliance: {overall_pct:.1f}%")
    print(f"   └─ Compliant: {total_compliant:,} movies")
    print(f"   └─ Overdue: {compliance_df['overdue_tmdb'].sum():,} movies")
    print(f"   └─ Never Updated: {compliance_df['never_updated_tmdb'].sum():,} movies")


📋 Refresh Strategy Compliance Analysis

             age_group  expected_refresh_days  total_movies  never_updated_tmdb  overdue_tmdb  compliant_tmdb  pct_compliant_tmdb  avg_days_since_update
        Recent (0-60d)                      5            34                   0             0              34               100.0                    4.4
         New (61-180d)                     14           126                   0             0             126               100.0                    4.0
Established (181-730d)                     30          1106                   0             0            1106               100.0                    4.0
    Mature (731-1825d)                     90          3153                   0             0            3153               100.0                    4.0
     Archived (1826+d)                    180         28115                   0             0           28115               100.0                    4.4

📊 Overall Compliance: 100.0%
   └─ Compl

In [12]:
# Visualize compliance by age group
if not compliance_df.empty:
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=compliance_df['age_group'],
        y=compliance_df['compliant_tmdb'],
        name='Compliant',
        marker_color='#10b981',
        text=compliance_df['pct_compliant_tmdb'].apply(lambda x: f'{x:.0f}%'),
        textposition='auto'
    ))

    fig.add_trace(go.Bar(
        x=compliance_df['age_group'],
        y=compliance_df['overdue_tmdb'],
        name='Overdue',
        marker_color='#ffa726',
        text=compliance_df['overdue_tmdb'],
        textposition='auto'
    ))

    fig.add_trace(go.Bar(
        x=compliance_df['age_group'],
        y=compliance_df['never_updated_tmdb'],
        name='Never Updated',
        marker_color='#ff6b6b',
        text=compliance_df['never_updated_tmdb'],
        textposition='auto'
    ))

    fig.update_layout(
        title='Refresh Compliance by Movie Age Group',
        xaxis_title='Age Group',
        yaxis_title='Number of Movies',
        barmode='stack',
        height=500,
        hovermode='x unified'
    )

    fig.show()

## 7. Data Freeze Pipeline

Tracks movies approaching the freeze threshold (3 consecutive unchanged refreshes + 1826+ days old).
Movies are frozen after multiple refreshes with no data changes to reduce API usage on stable content.

**Note**: If showing "No movies with consecutive unchanged refreshes", this is expected early in the collection process or after a system reset.

In [13]:
# Get consecutive unchanged refresh statistics
unchanged_df = get_consecutive_unchanged_stats()

if not unchanged_df.empty:
    print("\n🔒 Consecutive Unchanged Refresh Statistics\n")
    print(unchanged_df.to_string(index=False))

    # Highlight freeze candidates
    freeze_candidates = unchanged_df[
        (unchanged_df['consecutive_unchanged'] >= 2) &
        (unchanged_df['frozen_count'] < unchanged_df['old_enough_to_freeze'])
    ]

    if not freeze_candidates.empty:
        print(f"\n⚠️  Freeze Candidates (2+ unchanged, not yet frozen):")
        total_candidates = freeze_candidates['old_enough_to_freeze'].sum() - freeze_candidates['frozen_count'].sum()
        print(f"   └─ {total_candidates:,} movies approaching freeze threshold")
else:
    print("✅ No movies with consecutive unchanged refreshes")

✅ No movies with consecutive unchanged refreshes


In [14]:
# Visualize freeze pipeline
if not unchanged_df.empty:
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=unchanged_df['consecutive_unchanged'],
        y=unchanged_df['frozen_count'],
        name='Already Frozen',
        marker_color='#60a5fa'
    ))

    fig.add_trace(go.Bar(
        x=unchanged_df['consecutive_unchanged'],
        y=unchanged_df['old_enough_to_freeze'] - unchanged_df['frozen_count'],
        name='Freeze Candidates',
        marker_color='#fbbf24'
    ))

    fig.update_layout(
        title='Movie Freeze Pipeline - Consecutive Unchanged Refreshes',
        xaxis_title='Consecutive Unchanged Refreshes',
        yaxis_title='Number of Movies',
        barmode='stack',
        height=400,
        hovermode='x unified'
    )

    fig.add_vline(x=3, line_dash="dash", line_color="red",
                  annotation_text="Freeze Threshold", annotation_position="top")

    fig.show()

## 8. Data Freshness Heatmap

Shows how recently each year cohort was updated. Helps identify stale data requiring attention.

In [15]:
# Get data freshness by year
freshness_df = get_data_freshness_by_year()

if not freshness_df.empty:
    print("\n📅 Data Freshness by Release Year\n")

    # Show summary statistics
    recent_years = freshness_df[freshness_df['release_year'] >= 2020]
    if not recent_years.empty:
        print("Recent Years (2020+):")
        print(recent_years[['release_year', 'total_movies', 'avg_days_since_tmdb_update', 'most_recent_tmdb_update']].to_string(index=False))

    # Identify stale cohorts
    stale_threshold = 90  # days
    stale_years = freshness_df[freshness_df['avg_days_since_tmdb_update'] > stale_threshold]

    if not stale_years.empty:
        print(f"\n⚠️  Stale Data (>{stale_threshold} days since update):")
        print(f"   └─ {len(stale_years)} year cohorts")
        print(f"   └─ {stale_years['total_movies'].sum():,} total movies")
        print(f"\n   Top 5 Stalest Years:")
        top_stale = stale_years.nlargest(5, 'avg_days_since_tmdb_update')
        print(top_stale[['release_year', 'total_movies', 'avg_days_since_tmdb_update']].to_string(index=False))
else:
    print("No freshness data available")


📅 Data Freshness by Release Year

Recent Years (2020+):
 release_year  total_movies  avg_days_since_tmdb_update most_recent_tmdb_update
         2025           397                         4.0              2025-11-30
         2024           791                         4.0              2025-11-30
         2023           984                         4.0              2025-11-30
         2022          1083                         4.0              2025-11-30
         2021          1092                         4.0              2025-11-30
         2020          1038                         4.0              2025-11-30

Recent Years (2020+):
 release_year  total_movies  avg_days_since_tmdb_update most_recent_tmdb_update
         2025           397                         4.0              2025-11-30
         2024           791                         4.0              2025-11-30
         2023           984                         4.0              2025-11-30
         2022          1083             

In [16]:
# Visualize data freshness heatmap
if not freshness_df.empty and len(freshness_df) > 0:
    # Filter to years with significant data
    plot_df = freshness_df[freshness_df['release_year'] >= 1980].copy()

    if not plot_df.empty:
        # Create color scale: green (fresh) to red (stale)
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=plot_df['release_year'],
            y=plot_df['avg_days_since_tmdb_update'],
            mode='markers+lines',
            marker=dict(
                size=plot_df['total_movies'].apply(lambda x: min(max(x/10, 5), 20)),
                color=plot_df['avg_days_since_tmdb_update'],
                colorscale='RdYlGn_r',
                showscale=True,
                colorbar=dict(title="Days Since<br>Last Update"),
                line=dict(width=1, color='white')
            ),
            line=dict(width=1, color='lightgray'),
            text=plot_df.apply(lambda row: f"Year: {row['release_year']}<br>Movies: {row['total_movies']}<br>Avg Days: {row['avg_days_since_tmdb_update']:.0f}", axis=1),
            hovertemplate='%{text}<extra></extra>'
        ))

        # Add threshold line
        fig.add_hline(y=90, line_dash="dash", line_color="orange",
                      annotation_text="Stale Threshold (90 days)")

        fig.update_layout(
            title='Data Freshness by Release Year (marker size = movie count)',
            xaxis_title='Release Year',
            yaxis_title='Average Days Since Last TMDB Update',
            height=500,
            hovermode='closest'
        )

        fig.show()

## 9. API Usage & Capacity Planning

Estimates API usage and time to complete pending updates based on rate limits:
- **TMDB**: ~100K requests/day (conservative batch estimate)
- **OMDB**: 1K requests/day (single API key)

**Note**: OMDB pending includes movies with IMDb IDs lacking enrichment + movies where IMDb ID needs to be obtained from TMDB first.

In [17]:
# Get API usage estimates
api_estimates = get_api_usage_estimates()

print("="*70)
print(" " * 20 + "API USAGE & CAPACITY PLANNING")
print("="*70)

print(f"\n📊 Current State:")
print(f"   └─ Total Movies: {api_estimates['total_movies']:,}")
print(f"   └─ TMDB Enriched: {api_estimates['enriched_tmdb']:,} ({100.0 * api_estimates['enriched_tmdb'] / api_estimates['total_movies']:.1f}%)")
print(f"   └─ OMDB Enriched: {api_estimates['enriched_omdb']:,} ({100.0 * api_estimates['enriched_omdb'] / api_estimates['total_movies']:.1f}%)")

print(f"\n🔄 Pending Updates:")
print(f"   └─ TMDB Pending: {api_estimates['tmdb_pending']:,}")
print(f"      • Never Updated: {api_estimates['tmdb_never_updated']:,}")
print(f"   └─ OMDB Pending: {api_estimates['omdb_pending']:,}")
print(f"      • Never Updated: {api_estimates['omdb_never_updated']:,}")

print(f"\n⏱️  Estimated Completion Time (at current rates):")
print(f"   └─ TMDB: {api_estimates['tmdb_days_to_complete']:.1f} days")
print(f"      • Rate: {api_estimates['tmdb_rate_limit_per_day']:,} requests/day")
print(f"   └─ OMDB: {api_estimates['omdb_days_to_complete']:.1f} days")
print(f"      • Rate: {api_estimates['omdb_rate_limit_per_day']:,} requests/day")

if api_estimates['omdb_days_to_complete'] > 30:
    print(f"\n⚠️  OMDB backlog is substantial ({api_estimates['omdb_days_to_complete']:.0f} days)")
    print(f"   💡 Consider: Multiple API keys, prioritize recent years, or focus on movies with IMDb IDs first")

print("="*70)

                    API USAGE & CAPACITY PLANNING

📊 Current State:
   └─ Total Movies: 32,534
   └─ TMDB Enriched: 32,534 (100.0%)
   └─ OMDB Enriched: 6,663 (20.5%)

🔄 Pending Updates:
   └─ TMDB Pending: 0
      • Never Updated: 0
   └─ OMDB Pending: 51,164
      • Never Updated: 25,582

⏱️  Estimated Completion Time (at current rates):
   └─ TMDB: 0.0 days
      • Rate: 100,000 requests/day
   └─ OMDB: 51.2 days
      • Rate: 1,000 requests/day

⚠️  OMDB backlog is substantial (51 days)
   💡 Consider: Multiple API keys, prioritize recent years, or focus on movies with IMDb IDs first

                    API USAGE & CAPACITY PLANNING

📊 Current State:
   └─ Total Movies: 32,534
   └─ TMDB Enriched: 32,534 (100.0%)
   └─ OMDB Enriched: 6,663 (20.5%)

🔄 Pending Updates:
   └─ TMDB Pending: 0
      • Never Updated: 0
   └─ OMDB Pending: 51,164
      • Never Updated: 25,582

⏱️  Estimated Completion Time (at current rates):
   └─ TMDB: 0.0 days
      • Rate: 100,000 requests/day
   └─ O

In [18]:
# Visualize API backlog
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('TMDB Status', 'OMDB Status'),
    specs=[[{'type': 'pie'}, {'type': 'pie'}]]
)

# TMDB pie chart
fig.add_trace(
    go.Pie(
        labels=['Enriched', 'Pending', 'Never Updated'],
        values=[
            api_estimates['enriched_tmdb'],
            api_estimates['tmdb_pending'] - api_estimates['tmdb_never_updated'],
            api_estimates['tmdb_never_updated']
        ],
        marker_colors=['#10b981', '#fbbf24', '#ff6b6b'],
        hole=0.3
    ),
    row=1, col=1
)

# OMDB pie chart
fig.add_trace(
    go.Pie(
        labels=['Enriched', 'Pending', 'Never Updated'],
        values=[
            api_estimates['enriched_omdb'],
            api_estimates['omdb_pending'] - api_estimates['omdb_never_updated'],
            api_estimates['omdb_never_updated']
        ],
        marker_colors=['#10b981', '#fbbf24', '#ff6b6b'],
        hole=0.3
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='API Enrichment Status Distribution',
    height=400,
    showlegend=True
)

fig.show()

## 10. Collection Priority Recommendations

Actionable recommendations based on the analysis above.

In [19]:
# Generate prioritized recommendations
print("="*70)
print(" " * 18 + "📋 COLLECTION RECOMMENDATIONS")
print("="*70)

recommendations = []

# 1. Check for compliance issues
if not compliance_df.empty:
    overdue_total = compliance_df['overdue_tmdb'].sum()
    never_updated = compliance_df['never_updated_tmdb'].sum()

    if never_updated > 0:
        recommendations.append({
            'priority': 'HIGH',
            'category': 'Missing Data',
            'action': f'Initialize {never_updated:,} movies that have never been updated',
            'command': 'ayne tmdb refresh --limit 1000'
        })

    if overdue_total > 5000:
        recommendations.append({
            'priority': 'HIGH',
            'category': 'Refresh Compliance',
            'action': f'{overdue_total:,} movies are overdue for refresh',
            'command': 'ayne tmdb refresh --limit 5000'
        })

# 2. Check OMDB enrichment
omdb_pending = api_estimates['omdb_pending']
if omdb_pending > 0 and api_estimates['omdb_days_to_complete'] > 7:
    # Upgrade to HIGH priority if OMDB coverage is very low (< 30%)
    omdb_coverage = api_estimates['enriched_omdb'] / api_estimates['total_movies']
    priority = 'HIGH' if omdb_coverage < 0.3 else 'MEDIUM'
    recommendations.append({
        'priority': priority,
        'category': 'OMDB Enrichment',
        'action': f'Enrich {omdb_pending:,} movies with OMDB data (est. {api_estimates["omdb_days_to_complete"]:.0f} days, currently {omdb_coverage*100:.1f}% complete)',
        'command': 'ayne omdb enrich --max-movies 1000 --min-year 2020'
    })

# 3. Check for stale data
if not freshness_df.empty:
    stale_years = freshness_df[freshness_df['avg_days_since_tmdb_update'] > 90]
    if not stale_years.empty:
        stale_count = stale_years['total_movies'].sum()
        oldest_year = stale_years.loc[stale_years['avg_days_since_tmdb_update'].idxmax()]
        recommendations.append({
            'priority': 'MEDIUM',
            'category': 'Stale Data',
            'action': f'{stale_count:,} movies have not been updated in 90+ days',
            'command': f'ayne tmdb refresh --min-year {int(oldest_year["release_year"])} --max-year {int(oldest_year["release_year"])} --limit 500'
        })

# 4. Check data quality
if not quality_df.empty:
    tmdb_row = quality_df[quality_df['source_table'] == 'TMDB Movies']
    if not tmdb_row.empty:
        # Check budget/revenue completeness
        budget_col = [col for col in tmdb_row.columns if 'budget' in col.lower()]
        if budget_col and tmdb_row[budget_col[0]].iloc[0] < 50:
            recommendations.append({
                'priority': 'LOW',
                'category': 'Data Quality',
                'action': 'Low financial data completeness - consider additional sources',
                'command': 'Review The Numbers integration or add budget data sources'
            })

# 5. Check enrichment status
if not enrichment_df.empty:
    recent = enrichment_df[enrichment_df['release_year'] >= 2020]
    if not recent.empty:
        low_enrichment = recent[recent['pct_tmdb_enriched'] < 95]
        if not low_enrichment.empty:
            for _, row in low_enrichment.head(3).iterrows():
                recommendations.append({
                    'priority': 'HIGH',
                    'category': 'Recent Movies',
                    'action': f'{int(row["release_year"])}: Only {row["pct_tmdb_enriched"]:.0f}% enriched',
                    'command': f'ayne tmdb refresh --min-year {int(row["release_year"])} --max-year {int(row["release_year"])} --limit 1000'
                })

# Print recommendations
if recommendations:
    # Sort by priority
    priority_order = {'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
    recommendations.sort(key=lambda x: priority_order[x['priority']])

    for i, rec in enumerate(recommendations, 1):
        icon = '🔴' if rec['priority'] == 'HIGH' else '🟡' if rec['priority'] == 'MEDIUM' else '🟢'
        print(f"\n{i}. {icon} [{rec['priority']}] {rec['category']}")
        print(f"   Action: {rec['action']}")
        print(f"   Command: {rec['command']}")
else:
    print("\n✅ No urgent recommendations - database is in good shape!")

print("\n" + "="*70)

                  📋 COLLECTION RECOMMENDATIONS

1. 🔴 [HIGH] OMDB Enrichment
   Action: Enrich 51,164 movies with OMDB data (est. 51 days, currently 20.5% complete)
   Command: ayne omdb enrich --max-movies 1000 --min-year 2020



## 11. Custom Queries

Space for ad-hoc analysis and custom queries.

In [20]:
# Example: Top genres by count
query = """
SELECT
    genre,
    COUNT(*) as movie_count
FROM tmdb_movies,
    UNNEST(string_split(genres, ', ')) as t(genre)
WHERE genres IS NOT NULL AND genres != ''
GROUP BY genre
ORDER BY movie_count DESC
LIMIT 15
"""

genre_df = execute_custom_query(query)
print("\n🎬 Top 15 Genres by Movie Count:\n")
print(genre_df.to_string(index=False))

# Visualize
fig = px.bar(
    genre_df,
    x='movie_count',
    y='genre',
    orientation='h',
    title='Top Genres in Database',
    labels={'movie_count': 'Number of Movies', 'genre': 'Genre'}
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()



🎬 Top 15 Genres by Movie Count:

               genre  movie_count
              Comedy         2495
               Drama         2431
        Comedy,Drama          960
       Drama,Romance          954
         Documentary          786
              Horror          757
      Comedy,Romance          703
     Horror,Thriller          649
        Drama,Comedy          434
Comedy,Drama,Romance          423
      Drama,Thriller          389
       Drama,History          339
       Romance,Drama          275
      Romance,Comedy          271
     Action,Thriller          257


In [21]:
# Example: Movies added per month (last 12 months)
query = """
SELECT
    DATE_TRUNC('month', created_at) as month,
    COUNT(*) as movies_added
FROM movies
WHERE created_at >= CURRENT_DATE - INTERVAL '12 months'
GROUP BY month
ORDER BY month DESC
"""

monthly_df = execute_custom_query(query)
if len(monthly_df) > 0:
    print("\n📅 Movies Added per Month (Last 12 Months):\n")
    print(monthly_df.to_string(index=False))

    fig = px.line(
        monthly_df,
        x='month',
        y='movies_added',
        title='Database Growth: Movies Added per Month',
        markers=True
    )
    fig.update_layout(height=400)
    fig.show()
else:
    print("No recent movie additions found.")


📅 Movies Added per Month (Last 12 Months):

     month  movies_added
2025-11-01         32534


## Summary and Action Items

### Key Monitoring Areas

Review the cells above to identify:

#### 🎯 Immediate Actions
- **Section 1**: Overall database health and coverage
- **Section 3**: Movies requiring immediate updates
- **Section 10**: Prioritized collection recommendations

#### 📊 Performance Metrics
- **Section 2**: Enrichment rates by year
- **Section 4**: Data quality and completeness
- **Section 6**: Refresh strategy compliance

#### 🔍 Operational Insights
- **Section 5**: Recent collection activity trends
- **Section 7**: Freeze pipeline and stable movies
- **Section 8**: Data freshness across year cohorts
- **Section 9**: API capacity and timeline estimates

#### 💡 Strategic Planning
- **Low enrichment years** → Prioritize for data collection
- **Compliance gaps** → Adjust refresh workflows
- **Stale cohorts** → Target for updates
- **API backlogs** → Plan capacity or prioritization

---

*Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*